# 42 - Luu bao cao Notebook 41 len Drive

Chay **Run all** trong cung runtime Colab vua chay xong Notebook 41, truoc khi reset.
Neu notebook nay mo mot runtime moi, copy o code ben duoi vao mot o moi trong Notebook 41 dang mo va chi chay o do.
Khong chay lai Notebook 41. O nay chi sao luu bao cao, khong doc anh hay chay lai test.

In [ ]:
import hashlib
import json
from pathlib import Path
from shutil import copy2

src = Path('/content/fingerprint_research/native_pilot_nbis/reports')
dst = Path('/content/drive/MyDrive/FingerprintResearch/evaluations/native_triplet_heldout_test_v1')
reports = ('macro_metrics.csv', 'test_metrics.csv', 'paired_attempt_changes.csv')

def save_reports(source, destination):
    names = (*reports, 'completion.json')
    missing = [name for name in names if not (source / name).is_file()]
    if missing:
        raise FileNotFoundError(f'Thieu {missing}. Chay o nay trong runtime Notebook 41 cu, khong chay lai test.')
    marker = json.loads((source / 'completion.json').read_text(encoding='utf-8'))
    if marker['status'] != 'heldout_test_complete' or marker['variants'] != 8:
        raise RuntimeError('Khong tim thay marker Notebook 41 da hoan tat du 8 bien the.')

    def digest(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    expected = {name: marker['file_sha256'][name] for name in reports}
    expected['completion.json'] = digest(source / 'completion.json')
    for name in names:
        if digest(source / name) != expected[name]:
            raise RuntimeError(f'Checksum file goc khong khop: {name}')
        target = destination / name
        if target.exists() and digest(target) != expected[name]:
            raise FileExistsError(f'Tu choi ghi de file khac noi dung: {target}')

    destination.mkdir(exist_ok=True)
    for name in names:
        target = destination / name
        if not target.exists():
            pending = target.with_suffix(target.suffix + '.part')
            copy2(source / name, pending)
            if digest(pending) != expected[name]:
                raise RuntimeError(f'Sao luu that bai (checksum khong khop): {pending}')
            pending.replace(target)
    print('Da luu va xac minh 4 bao cao:', destination)

if not (src / 'completion.json').is_file():
    raise FileNotFoundError('Khong thay bao cao. Chay o nay trong runtime Colab cua Notebook 41 truoc khi reset.')
if not dst.parent.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if not dst.parent.is_dir():
    raise FileNotFoundError(f'Khong tim thay thu muc evaluations tren Drive: {dst.parent}')
save_reports(src, dst)